# 1. What is clustering (really)?

**Informal version:**
- You have millions of points (customers, images, whatever).
- You don’t want to look at all of them one by one (because you’re not insane).
- Clustering = group similar points together so you can reason about groups instead of single points.

So:
- Input: data points, no labels.
- Output: clusters (groups of points).
- Goal: summarize / understand the structure of the data, not predict a specific y.

Important:
- There’s no unique “right” clustering.
- 2 clusters might make sense.
- 3 clusters might also make sense.
- 10 clusters? Sometimes still reasonable.
- It’s more like: “Is this clustering useful for my task?” than “Is this clustering correct?”.

Your text says “no way to evaluate a cluster” → that’s a bit extreme.
- There’s no ground-truth label usually.
- But we do have:
  - Internal metrics (silhouette score, etc.).
  - External metrics (if labels exist, like Adjusted Rand Index).
- Still, evaluation is more “qualitative + task-driven” than in supervised learning.

⸻

## 2. Typical uses

Two big buckets:
1. Understanding / exploration  
   - Group similar documents (e.g., news articles by topic).  
   - Group genes/proteins that behave similarly.  
   - Group stocks with similar price movements.  
   - Group customers with similar behavior (classic “customer segments”).  

2. Summarization  
   - Instead of 10M rows, look at 20 clusters (each cluster ~ “typical pattern”).  
   - Good for:
     - reporting,  
     - reducing dimensionality,  
     - compressing information.  

⸻

## 3. What is a “clustering”? What is a “cluster”?

- Cluster = set of points grouped together.
- Clustering = a set of clusters.

That’s it. Nothing more mystical than that.

⸻

## 4. Types of clusterings

### 4.1 Partitional vs Hierarchical

- Partitional clustering  
  - Split data into non-overlapping subsets.  
  - Each point belongs to exactly one cluster.  
  - Example: k-means: you say “k=3” and get 3 clusters, done.  

- Hierarchical clustering  
  - Organizes clusters in a tree.  
  - You can “cut” the tree at different levels:
    - high level → few big clusters,
    - low level → many smaller clusters.
  - Useful when you want the multi-scale structure (clusters inside clusters).

⸻

### 4.2 Exclusive vs Non-exclusive

- Exclusive (hard) clustering  
  - Each point belongs to one cluster only.  
  - Example: k-means assigns each point to exactly one centroid.  

- Non-exclusive (overlapping) clustering  
  - Points can belong to multiple clusters.  
  - Example: a movie can be both “Action” and “Comedy”.

⸻

### 4.3 Fuzzy vs Non-fuzzy

- Non-fuzzy (hard):  
  - Point either is in a cluster or is not: 0 or 1.  

- Fuzzy clustering:  
  - Each point belongs to every cluster with a weight between 0 and 1.  
  - Weights over all clusters sum to 1.  
  - Example: point x:
    - cluster 1: 0.7  
    - cluster 2: 0.2  
    - cluster 3: 0.1  
  - Interpretation: “x is mostly in cluster 1, but sort of related to others.”  

- Probabilistic clustering is very similar:  
  - Think of those weights as probabilities.

⸻

### 4.4 Partial vs Complete

- Complete clustering  
  - Every point is assigned to some cluster.  

- Partial clustering  
  - Some points are left out:
    - noise,
    - outliers.
  - Example: density-based methods (DBSCAN) often mark some points as “noise”.

⸻

### 4.5 Homogeneous vs Heterogeneous

- Homogeneous clusters  
  - All clusters have similar size/shape/density (this is the fairy-tale world where k-means works well).  

- Heterogeneous clusters  
  - Clusters differ a lot:
    - some big, some small,  
    - some dense, some sparse,  
    - some spherical, some weird shapes.  

Many real datasets are like this → simple algorithms struggle.

⸻

## 5. Different notions of “what is a cluster?”

This is where definitions change depending on the method.

### 5.1 Well-separated clusters (naive idea)

A cluster is a set of points such that any point in a cluster is closer to every other point in the cluster than to any point outside.

- Intuition: clusters are tight blobs far from each other.
- Reality: too strict, often not true in real data.

⸻

### 5.2 Center-based clusters

A cluster is a set of points closer to the center of that cluster than to the center of any other cluster.

- “Center” can be:
  - Centroid: mean of all points in the cluster.
  - Medoid: actual data point that is most “central”.

This is exactly how k-means thinks:
- For each point:
  - compute distance to each centroid,
  - assign to the nearest centroid.

⸻

### 5.3 Contiguity-based (connectivity-based) clusters

A cluster is a set of points such that each point is closer to at least one other point in the same cluster than to any point outside the cluster.

- Think of chaining:
  - point A close to B,
  - B close to C,
  - C close to D,
  - so A, B, C, D form a cluster, even if A and D are not super close directly.
- This is related to nearest-neighbor chains, graph-based clustering, and some hierarchical methods.

⸻

### 5.4 Density-based clusters

A cluster is a dense region of points, separated by areas of low density.

- Idea:
  - Where points are very dense → cluster.
  - Areas with few points → “holes” or noise between clusters.
- Good when:
  - clusters are irregular or intertwined,
  - you have outliers.
- Classic algorithm: DBSCAN.

⸻

## 6. Main families of clustering algorithms

- K-means and variants  
  - Center-based, partitional, hard.  
  - Assumes spherical-ish clusters, similar sizes.  

- Hierarchical clustering  
  - Builds a tree (dendrogram).  
  - Agglomerative (bottom-up) or divisive (top-down).  

- Density-based clustering  
  - DBSCAN, HDBSCAN, etc.  
  - Finds arbitrary-shaped clusters & marks noise.

---

---

> # **1. What is k-means clustering?**

- Type: partitional, center-based, hard clustering.
- You choose a number of clusters K.
- The algorithm:
  - Associates each cluster with a centroid (a center point).
  - Assigns each data point to the cluster with the closest centroid (usually Euclidean distance).

**Key idea:**
You approximate your dataset with K “representative” points (the centroids), and each data point “belongs” to one of them.

**Big drawback:**
You must choose K, and that’s often the hardest part.

⸻

## 2. Basic k-means algorithm

**Intuition:**
1. Start with K initial centroids (typically random).
2. Repeat two steps until it “stops changing”:
   1. **Assignment step**  
      Assign each point to the cluster whose centroid is closest (according to some distance, usually Euclidean).
   2. **Update step**  
      For each cluster, recompute its centroid as the mean of all points currently assigned to it.
3. Stop when:
   - Centroids don’t move anymore, or
   - Very few points change cluster between iterations, or
   - You hit a max number of iterations.

**Pseudo-version of your text:**
Select K points as the initial centroids.  
repeat  
&emsp;Assign each point to the nearest centroid → form K clusters  
&emsp;Recompute the centroid of each cluster (mean of its points)  
until centroids don’t change (or assignments stabilize)

⸻

## 3. Practical details

- **Initial centroids:**
  - Often random → bad choices can give horrible clusters.
  - Different random seeds → different results.
- **Centroid definition:**
  - Typically the mean of all points in a cluster.
  - The mean minimizes the sum of squared Euclidean distances to points in that cluster.
- **Distance:**
  - Usually Euclidean distance.
  - K-means is basically “minimize squared Euclidean distances to cluster means”.
- **Convergence:**
  - Most improvement happens in the first few iterations.
  - It always converges to a local minimum of the objective, not necessarily the global best.
- **Time complexity:**  
  O(n \cdot K \cdot I \cdot d)  
  where:
  - n = number of points,
  - K = number of clusters,
  - I = number of iterations,
  - d = number of features.  

Linear in the number of points → very efficient, which is why people love it despite its issues.

⸻

## 4. Evaluating k-means: Sum of Squared Error (SSE)

Most common internal measure:
- For each point, error = distance to its cluster centroid.
- SSE = sum of squared distances of each point to its cluster centroid.

**Formula:**

\text{SSE} = \sum_{i=1}^K \sum_{x \in C_i} \|x - m_i\|^2  

- C_i = cluster i,
- m_i = centroid (mean) of cluster i.

**Properties:**
- You can show that the mean is the point that minimizes this SSE for a given cluster.
- Given two clusterings, the one with lower SSE is “better” (for this metric).
- But: SSE almost always decreases if you increase K (more clusters → smaller distances).
- So simply increasing K will usually reduce SSE, but that’s just overfitting.

⸻

## 5. The initial centroid problem (and fixes)

Random initialization is fragile:
- Different seeds → different solutions.
- Bad initial centroids → bad local minimum.

**Heuristic fixes:**
1. **Multiple runs**
   - Run k-means several times with different random init.
   - Keep the solution with the lowest SSE.
   - Helps, but no guarantee; just improves odds.
2. **Use hierarchical clustering for initialization**
   - On a sample of the data, run hierarchical clustering.
   - Use its cluster centers as better starting points for k-means.
3. **Select spread-out centroids (k-means++)**
   - Pick first centroid randomly.
   - Then choose new centroids probabilistically farther from existing ones.
   - Goal: initial centroids that are far apart in the data space → better starting configuration.
4. **Post-processing / bisecting k-means**
   - See below; also used to mitigate initialization issues.

⸻

## 6. Choosing K: the elbow method

We use SSE vs K:
1. Run k-means for different K (e.g. 2, 3, 4, …).
2. For each K, compute SSE.
3. Plot SSE(K).

**Observation:**
- SSE always decreases as K increases.
- But usually:
  - For small K: SSE drops a lot with each extra cluster.
  - After some point, extra clusters only slightly reduce SSE.

The “elbow” (or knee) is:
- The point where adding more clusters doesn’t give big SSE improvements anymore.
- That K is a reasonable trade-off:
  - not too few clusters (underfitting),
  - not too many (overfitting/uselessly complex).

⸻

## 7. Handling empty clusters

In some iterations, k-means can create a cluster with no points assigned.

**Fix strategies:**
1. **Steal the worst point**
   - Find the point that contributes the most to SSE (farthest from its current centroid).
   - Make that point the centroid of the empty cluster.
2. **Steal from the worst cluster**
   - Find the cluster with the highest SSE (most “spread”).
   - Take a point from there to form a new cluster.
3. **Multiple empty clusters**
   - Repeat the above for each empty cluster.

Goal: avoid having “dead” clusters with no points.

⸻

## 8. Pre-processing and post-processing

**Pre-processing**
- Normalize/standardize features:
  - k-means is distance-based, so scales matter a lot.
  - Without normalization, one big-scale feature dominates.
- Remove outliers:
  - Outliers can drag centroids far away and ruin clusters.

**Post-processing**  
Once k-means finishes:
- Remove tiny clusters:
  - They may represent outliers or noise.
- Split loose clusters:
  - Clusters with high SSE (very spread) might actually be multiple clusters.
  - You can re-run k-means within that cluster to split it.
- Merge very close clusters:
  - If two clusters are too close and both compact, merge them.

These steps can also be used during clustering (e.g., bisecting approaches).

⸻

## 9. Bisecting k-means

**Idea:** build clusters top-down, splitting one at a time.

**Process:**
1. Start with all points in one cluster.
2. Iterate until you have K clusters:
   1. Pick a cluster to split (often the one with highest SSE).
   2. Run k-means with K = 2 only on that cluster (i.e., bisect it).
   3. Repeat the K=2 run multiple times with different inits, keep the split with lowest SSE.
   4. Replace the original cluster with the two new clusters.

**Advantages:**
- Builds a hierarchical structure (tree of splits).
- Less sensitive to bad initialization than “flat” k-means on all data at once.
- Often better when clusters differ in size or density.

⸻

## 10. Limitations of k-means (very important)

k-means struggles when:
1. Clusters have different sizes
   - Big vs small clusters → centroids get pulled and assignments get weird.
2. Clusters have different densities
   - Dense vs sparse clusters → a single radius/shape doesn’t fit all.
3. Clusters are not globular (non-spherical)
   - Think of:
     - elongated shapes,
     - “rings” around a center,
     - intertwined crescents.
   - k-means assumes “blob-like” clusters around means.
4. Data contains outliers
   - Means are extremely sensitive to extreme points.
   - Outliers can:
     - attract centroids,
     - force extra clusters just to “explain” them.

One partial hack: use many clusters so k-means finds small “pieces” of the true clusters, then try to combine them. But it’s messy and not always satisfying.

That’s why for complex shapes / noise you often move to density-based methods (DBSCAN, etc.) or more advanced clustering.

---

---

> # 1. What is hierarchical clustering?

- It builds a hierarchy of clusters, not just one flat partition like k-means.
- Output is a tree of clusters, not just “K groups”.

You visualize it with a:

> **Dendrogram**

- A dendrogram is a tree-like diagram that shows:
  - which clusters/points are merged together,
  - and at what “distance” (or dissimilarity) the merge happened.
- You start with many small clusters and end with one big cluster:
  - bottom of the tree → each leaf is a single point,
  - as you go up → merges happen,
  - top → all points merged into a single cluster.

How to get K clusters?

- Just “cut” the dendrogram horizontally at some level:
  - cut high → few big clusters,
  - cut lower → more, smaller clusters.
- No need to fix K in advance: you can choose it after you see the tree.

⸻

2. Two main flavors: agglomerative vs divisive

**Agglomerative (bottom-up)**

- Start: each point is its own cluster.
- Iteratively:
  1. Find the two closest clusters (according to some distance between clusters).
  2. Merge them into a single cluster.
  3. Update the distance/proximity matrix.
- Stop: when you have 1 cluster (or you stop early with K clusters).

This is the one people usually mean when they say “hierarchical clustering”.

**Divisive (top-down)**

- Start: all points in one big cluster.
- Iteratively:
  1. Pick a cluster.
  2. Split it into 2 smaller clusters (e.g. by a k-means with K=2, or some other method).
- Stop: when each cluster has 1 point, or when you have K clusters.

Used less often than agglomerative in basic courses, but same idea in reverse.

⸻

3. Proximity matrix + complexity

Hierarchical (agglomerative) clustering usually works with a proximity (distance) matrix:

- For n points:
  - You build an n×n matrix of pairwise distances.
- Algorithm:
  1. Compute the proximity matrix (all pairwise distances).
  2. Let each point be its own cluster.
  3. Repeat:
     - merge the two closest clusters,
     - update the proximity matrix for the new cluster.
  4. Continue until only one cluster remains.
- Complexity: O(n²) in both time and memory.
  - You need to store (and update) the n×n distance matrix.
  - That’s why hierarchical clustering is expensive for large n.

⸻

4. The key question: distance between clusters, not points

Once you start merging points into clusters, you need:

“What is the distance between cluster A and cluster B?”

Different answers → different hierarchical algorithms (linkage methods).

Common definitions:

1. Single link (MIN)  
2. Complete link (MAX)  
3. Group average  
4. Centroid / Ward (just mentioned in your text)

⸻

5. Single link (MIN)

**Definition:**

- Distance between cluster A and B = the minimum distance between any pair of points (one in A, one in B).

$d_{\text{single}}(A, B) = \min_{x \in A, y \in B} d(x, y)$

**Interpretation:**

- “How close can I get if I pick one point in A and one in B?”
- It only looks at one pair of points → the closest pair.

**Strengths:**

- Can handle non-elliptical, weird shapes:
  - If clusters are like chains or long “snakes”, single-link can join them correctly,
  - because it only needs a chain of close neighbors to connect everything.

**Limitations:**

- Very sensitive to noise / outliers:
  - If there is even one noisy point that is close to another cluster,
  - the MIN distance becomes small,
  - clusters may get wrongly merged because of a single outlier.
- It’s “optimistic”:
  - “If I find one close pair, they’re friends now.”

⸻

6. Complete link (MAX)

**Definition:**

- Distance between cluster A and B = the maximum distance between any pair of points (one in A, one in B).

$d_{\text{complete}}(A, B) = \max_{x \in A, y \in B} d(x, y)$

**Interpretation:**

- “What is the worst-case distance between points in A and B?”
- It looks at the farthest pair between the two clusters.

**Important clarification:**

- For each pair of clusters, you compute their “distance” as this MAX.
- But when choosing which clusters to merge, you still pick the pair with the smallest of those MAX distances.
- MAX is only the definition of cluster–cluster distance,
- but we still choose the closest pair of clusters according to that.

**Strengths:**

- Less sensitive to noise and outliers than single link:
  - One noisy point far away will blow up the MAX distance.
  - That makes two clusters look far apart → they’re less likely to be merged prematurely.

**Limitations:**

- Tends to break large clusters:
  - Because it worries about the farthest pair, it prefers compact, “tight” clusters.
- Biased toward globular (ball-shaped) clusters:
  - Long or irregular clusters lead to large MAX distances,
  - so they tend to be split.

⸻

7. Group average linkage

**Definition:**

- Distance between clusters A and B = average of all pairwise distances between points in A and B.

$d_{\text{avg}}(A, B) =\frac{1}{|A| \cdot |B|}\sum_{x \in A} \sum_{y \in B} d(x, y)$

**Interpretation:**

- “On average, how far are points in A from points in B?”
- Uses all pairs, but averages them.

**Why care about average vs total?**

- If you use total distance (sum instead of average), large clusters automatically have large values, just because they have many points.
- Average distance makes the measure less biased towards cluster size.

**Strengths:**

- Often a compromise between single and complete link:
  - Less sensitive to outliers than single link (since it averages).
  - Less obsessed with worst-case than complete link.

**Limitations:**

- Still biased toward globular clusters:
  - It still “likes” clusters where distances within/between are homogeneous and not elongated.

---

# 1. **Ward’s method** (hierarchical “k-means-style” linkage)

**What it is:**
- It’s a way to define “distance” between two clusters in hierarchical clustering.
- It is the default in scikit-learn’s `AgglomerativeClustering` (`linkage="ward"`).
- Idea: merge the pair of clusters that causes the smallest increase in total SSE (sum of squared errors).

**Intuition:**
- Each cluster has a centroid (mean).
- SSE of a clustering = sum over all clusters of:
  “squared distance of each point to its cluster centroid”.
- If you merge two clusters A and B:
  - the SSE will go up (you’re forcing two groups to share one centroid),
  - Ward’s method chooses the merge that increases SSE the least at each step.

So at every merge step:
1. Consider all pairs of clusters (A, B).
2. For each pair, pretend to merge them, compute how much SSE would increase.
3. Merge the pair with the smallest increase.

**Properties:**
- Very similar in spirit to group average if you use squared distances.
- Less sensitive to noise than single-link: one weird close pair doesn’t dominate.
- Still biased toward globular / compact clusters (like k-means).
- It’s basically the hierarchical analogue of k-means:
  - both try to minimize SSE,
  - both like compact, ball-shaped clusters.
- You can also use Ward’s output to initialize k-means:
  - e.g. take the centroids of the final hierarchical clusters as initial centers.

⸻

2. Hierarchical clustering complexity (why it sucks on big data)

For agglomerative hierarchical clustering with a proximity matrix:
- You store all pairwise distances → that’s an n×n matrix.

**Space complexity:**
- O(n²) space:
  - with n points, number of pairwise distances ≈ n(n−1)/2 → quadratic.

**Time complexity (naive):**
- O(n³) in many straightforward implementations:
  - There are n−1 merge steps.
  - At each step you:
    - search the distance matrix for the closest pair (O(n²)),
    - update distances involving the new merged cluster.
  - Roughly: n steps × O(n²) work per step ≈ O(n³).

**Improved:**
- With clever data structures and update rules, you can get around O(n² log n) for some linkages.
- But the n² memory is still there. That alone kills you for millions of points.

**What the slides hint with “run K-means first and then hierarchical”:**
- On very large datasets you might:
  1. Run k-means (or mini-batch k-means) to compress data into, say, M centroids (M ≪ n).
  2. Run hierarchical clustering on those M centroids.
- You’re effectively clustering “clusters”:
  - Step 1: get K₀ small clusters cheaply with k-means.
  - Step 2: hierarchically cluster those K₀ centroids.
- Trade-off: you lose some information, but make hierarchical clustering feasible.

⸻

> # 3. **DBSCAN: density-based clustering**

**Goal:**
- Find dense blobs of points = clusters.
- Declare sparse, lonely points = noise.

**Two key parameters:**
- `eps` (ε):
  - distance threshold: “how close is considered a neighbor”.
- `minPts`:
  - minimum number of neighbors within eps to be considered dense.

**Definitions:**
- Core point:
  - a point that has at least minPts points (including itself) within distance eps.
  - i.e. enough neighbors → clearly inside a cluster.
- Border point:
  - not a core point (fewer than minPts neighbors),
  - but lies within eps of a core point.
  - on the “edge” of a cluster.
- Noise point:
  - not core, not border → too lonely.
  - treated as outlier.

DBSCAN does basically two things:
1. Find dense groups → clusters.
2. Throw away noise.

That’s the whole story.

⸻

> # 4. DBSCAN algorithm, step by step

**Parameter choice:**
- `eps` = neighborhood radius.
- `minPts` = required neighbors to be a core point.
- Example: `eps = 1`, `minPts = 3`:
  - a point with ≥ 3 neighbors within distance 1 → core.

**Algorithm:**
0. Mark all points as “unvisited”.

1. Take an unvisited point P, mark it visited.  
   Find all points within distance eps → its neighborhood N(P).

2. If |N(P)| < minPts:
   - P is not core → temporarily mark as noise (may later become border if touched by another cluster).

3. If |N(P)| ≥ minPts:
   - P is a core point:
     - start a new cluster C,
     - add P to C,
     - add all points in N(P) to C.

4. Cluster expansion:
   - For each point Q in N(P):
     - If Q is unvisited:
       - mark it visited,
       - find its neighbors N(Q) within eps,
       - if |N(Q)| ≥ minPts → Q is core:
         - add N(Q) to the cluster (expand the frontier).
     - If Q is not yet assigned to any cluster:
       - assign Q to C.

5. Repeat:
   - pick another unvisited point and go again.
   - stop when all points are visited.

That pseudo-code blob you pasted with `current_cluster_label` is just an implementation detail of this same idea.

⸻

> # 5. When DBSCAN works well vs when it fails

**Works well when:**
- Data has clusters of roughly similar density.
- Clusters can be:
  - arbitrarily shaped (rings, spirals, snake-like),
  - not globular.
- There is noise / outliers you want to detect and ignore:
  - DBSCAN explicitly labels some points as noise → big win over k-means.

**Fails / struggles when:**

1. Varying densities:
   - Suppose:
     - one cluster is very dense,
     - another is much sparser.
   - With a single (eps, minPts):
     - either eps is small: you split the sparse cluster or mark many of its points as noise,
     - or eps is large: you start merging everything together or swallowing noise.
   - DBSCAN assumes a single global density scale → not great when densities vary.

2. High-dimensional data:
   - In high dimensions, distances tend to “concentrate”:
     - nearest and farthest neighbors have similar distances,
     - density estimation via eps radius becomes meaningless.
   - Choosing a good eps is extremely hard, clusters don’t stand out as dense balls anymore.

⸻

> # 6. Choosing eps and minPts: k-distance plot (the thing that was confusing)

**Idea:**
- Points that belong to a “nice” cluster should have their k-th nearest neighbor at roughly similar distance (within cluster scale).
- Points that are noise tend to have larger k-distance (far from others).

Here k is typically set to minPts (or minPts−1), e.g. k = 4 or 5.

**Procedure:**
1. Choose minPts (e.g. 4):
   - rule of thumb: minPts ≈ dimensionality + 1, or at least 3–5.

2. For every point i:
   - compute the distance to its k-th nearest neighbor (k = minPts).
   - call this dₖ(i).

3. Collect all these distances {dₖ(i)}.

4. Sort them in ascending order.

5. Plot sorted dₖ(i) vs point index.

**What you see in the k-distance plot:**
- On the left: points in dense regions:
  - their k-th nearest neighbor is close → small dₖ.
- At some point, the curve starts to rise sharply:
  - those are likely noise points, whose k-th neighbor is far away.
- The “elbow” (where the slope changes) is a good candidate for eps:
  - pick eps near the start of the steep region.

So:
- minPts: fixed by you (small integer).
- eps: chosen from the elbow in the sorted k-distance curve.

That’s what the slide means by:
> “plot sorted distance of every point to its k-th nearest neighbor.”

---

---

> **1. Why cluster validity is even a thing**

In classification (supervised):
- You have true labels → cat, dog, horse.
- You can say:
  - “Accuracy = 90%”
  - “Recall = 95%”
  - “Precision = 88%”
- Life is simple.

In clustering (unsupervised):
- No true labels.
- You have no idea what the “real” clusters are.
- So “Is this clustering good?” is not obvious at all.

Also:
- Different humans can look at the same data and imagine different “natural” clusters.
- Hence the classic line: “clusters are in the eye of the beholder”.

Still, we must evaluate clusters, or we’re just doing numerically expensive astrology.

Why evaluate?
1. To avoid finding patterns in pure noise.
2. To compare different algorithms (e.g. k-means vs DBSCAN vs hierarchical).
3. To compare two full clusterings on the same data.
4. To compare individual clusters (e.g. “Should I merge cluster 3 and 4?”).

Same idea restated in the slides:
- We want to:
  - avoid fake patterns,
  - compare algorithms,
  - compare two sets of clusters,
  - compare two single clusters.

⸻

2. Different aspects of cluster validation

The slides list 5 “what are we actually trying to check?” questions:

1. Clustering tendency  
   - Is there any structure at all, or is the data basically random?  
   - If everything is just noise, any clustering is meaningless.

2. Comparison with external labels  
   - If you do have some external ground truth (e.g., known categories), you can see:  
   - Did the clustering recover those?

3. Internal goodness, no external info  
   - Evaluate how well the clustering fits the data using only:  
     - within-cluster distances,  
     - between-cluster distances.

4. Compare two clusterings  
   - You might have:  
     - clustering A (k-means),  
     - clustering B (DBSCAN),  
     - same data.  
   - Which fits data better? (according to some index)

5. Determine the “right” number of clusters  
   - e.g. choose K in k-means or where to cut a dendrogram.

And for 2, 3, 4 you can either:
- Evaluate the entire clustering, or
- Evaluate specific clusters.

⸻

> #### 3. Three big families of validity indices


We’ve got 3 main classes:

1. External indices
   - You have true labels.
   - You measure how much cluster labels match those true labels.
   - Examples:
     - Entropy:
       - High entropy → cluster contents are mixed (bad).
       - Low entropy → cluster is “pure” (mostly one class).
     - Purity:
       - For each cluster, take the majority class fraction.
       - Combine across clusters → high purity means clusters align with real labels.
   - Use-case: “Did my clustering recover the known classes?”

⸻

> #### 2. Internal indices

- No external labels.
- Only use the data + cluster assignments.

You measure:
- **Cohesion**: how tight are points inside clusters?
- **Separation**: how far apart are clusters?

Examples:
- SSE (Sum of Squared Errors):
  - For each cluster:
    - distance of each point to the cluster centroid, squared,
    - sum all.
  - Lower SSE → tighter clusters.
- Cluster cohesion / cluster separation (generic names).
- Silhouette score:
  - Points get a score between -1 and 1, combining cohesion and separation.
  - Closer to 1 → good assignment.

Analogy: you don’t know the correct answers (labels), so you judge by “do these groups look compact and separated?”.

⸻

> #### 3. Relative indices

- You want to compare multiple clusterings.

Examples:
- k-means with K=3 vs K=5,
- or k-means vs DBSCAN.

You don’t necessarily know which is “correct”, but you can say which one is “better” according to some index.

Examples:
- Rand Index / Adjusted Rand Index:
  - Compare two different partitions of the same data.
  - Higher ARI → more similar.

Also:
- In practice, people often reuse an internal or external index in a relative way:
  - e.g. compare SSE of k-means with K=3 vs K=5.

Slides summary:
- External: entropy, purity, etc.
- Internal: SSE, cohesion, separation, silhouette.
- Relative: Rand, Adjusted Rand, or reuse internal/external metrics to compare clusterings.

⸻

4. Cohesion and separation in more detail

> # 4.1 **Cluster cohesion** (within-cluster SSE)

Question:
- “Are points in the same cluster close to their centroid or scattered everywhere?”

Mathematically:
- For each cluster $C_i$ with centroid $m_i$:
  - Sum over $x \in C_i$ of $\|x - m_i\|^2$.
- Then sum this over all clusters:

$\text{WSS (within-cluster sum of squares)} = \sum_i \sum_{x \in C_i} \|x - m_i\|^2 $

Interpretation:
- Small WSS → points tightly packed around their centroids → good cohesion.
- Large WSS → clusters are loose blobs → poor cohesion.

Graphically: imagine each cluster as a group of friends around their “leader”.  
If friends stand close to their leader → small WSS, nice cluster.  
If they’re spread across the room like they hate each other → big WSS, trash cluster.

⸻

> # 4.2 **Cluster separation** (between-cluster SSE)

Now instead we ask:
- “How far apart are clusters from each other?”

Classic measure: BSS (between-cluster sum of squares).

Let:
- $m$ = global mean of all data.
- $m_i$ = centroid of cluster $C_i$.
- $|C_i|$ = number of points in cluster $C_i$.

Then:

$\text{BSS} = \sum_i |C_i| \cdot \| m_i - m \|^2$

Interpretation:
- If cluster centroids are far from the global mean and from each other → large BSS → good separation.
- If all centroids are near the global mean → small BSS → clusters overlap a lot.

Informal party analogy:
- Two groups of friends standing far apart in the room → high separation.
- Everyone crowded in the middle → low separation (your clusters are basically fake).

⸻

> # 5. Silhouette: a very handy internal score

Silhouette is defined per point, then averaged.

For each point i:
- $a(i)$: average distance from i to all other points in its own cluster.
  - Measures cohesion for point i.
  - Smaller is better.
- $b(i)$: for every other cluster:
  - compute average distance from i to points in that other cluster,
  - take the minimum of these averages.
  - This is “the next best alternative cluster” for point i.

Then:

$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$

Properties:
- $s(i) \in [-1, 1]$.
- In practice, usually between 0 and 1 if things aren’t terrible.
- Interpretation:
  - Close to 1 → well clustered (much closer to its own cluster than to others).
  - Around 0 → on the border between two clusters.
  - Negative → probably in the wrong cluster.

You can average $s(i)$:
- over all points in a cluster → quality of that cluster.
- over all points in the dataset → quality of whole clustering.

⸻

6. External measures: entropy & purity


- Suppose you do have true class labels.

**Purity:**
- For each cluster:
  - find the most frequent true class in that cluster,
  - purity(cluster) = fraction of points in that cluster that belong to that class.
- Overall purity = weighted average over clusters.
- High purity → each cluster is dominated by a single class.

**Entropy:**
- For each cluster, look at the distribution of true classes in it.
- High entropy = very mixed (e.g. 33% A, 33% B, 34% C).
- Low entropy = mostly one class (e.g. 95% A, 5% noise).
- Lower entropy → cleaner cluster.

Both are external: they only make sense if you know “real” labels.

⸻

> # 7. Rand Index (RI)

Goal:
- Compare two clusterings of the same data:
  - e.g. ground-truth classes vs your clustering.
  - or clustering A vs clustering B.

Idea:
- Look at all pairs of points (i, j).
- For each pair, check:
  - Are they in the same cluster or different clusters in clustering 1?
  - Same/different in clustering 2?

Define:
- $f_{00}$: number of pairs of objects having a different class and a different cluster
- $f_{01}$: number of pairs of objects having a different class and the same cluster
- $f_{10}$: number of pairs of objects having the same class and a different cluster
- $f_{11}$: number of pairs of objects having the same class and the same cluster

Rand Index:

$\text{RI} = \frac{f_{00} + f_{11}}{f_{00} + f_{01} + f_{10} + f_{11}}$

Interpretation:
- $f_{11}$: pairs that both clusterings agree belong together.
- $f_{00}$: pairs that both clusterings agree should be apart.
- RI = fraction of pairs where the two partitions agree (same vs different).

Values:
- RI close to 1 → clusterings agree a lot.
- RI near 0.5-ish → more or less random (depending on class structure).
- Adjusted Rand Index (ARI) corrects for chance agreement.

⸻

8. Final philosophical punchline

“The validation of clustering structures is the most difficult and frustrating part of cluster analysis.
Without a strong effort in this direction, cluster analysis will remain a black art accessible only to those true believers who have experience and great courage.”

Translation:
- Clustering is easy to run, hard to trust.
- You absolutely need validation tools (internal/external/relative indices, domain sanity checks).
- Otherwise you’re just drawing circles around noise and giving them fancy names.